# Занятие 6, пара 2. SQL. Первое знакомство


Автор ноутбука - Дуркин Анатолий Альбертович

Старший преподаватель кафедры прикладной математики и компьютерных наук СГУ им. Питирима Сорокина

Замечания, предложения, идеи, вопросы, связь с автором:
- anatoliy.durkin@mail.ru
- Telegram - @AnatoDu

Больше информации и материалов на канале автора: https://t.me/code_matan_ai


**Как проходить тетрадь.** Основные примеры работают без ответов «Ваш ход».
        Если пропускаете упражнение, пропустите также его `check(...)`. Продолжайте следующий обычный раздел.
        Сохраняйте свои попытки в отдельной копии. Проверки помогают разобраться, но не выставляют баллы.
        Интернет, Git и регистрация для основной работы не нужны.


## Та же таблица, другой язык вопросов

Pandas просит нас записать последовательность действий. SQL позволяет описать, какие строки
и столбцы нужны в результате. Мы будем задавать **те же вопросы тем же данным**, чтобы новый
язык не смешался с новой предметной областью.

SQL — язык запросов, SQLite — программа, которая их выполняет. База может содержать несколько
связанных таблиц. Сегодня она живёт в памяти Python: сервер, пароль и установка DuckDB не нужны.
Это первое знакомство, не полный курс баз данных. DuckDB остаётся отдельным инструментом будущего блока.

Нужны: понятия строки, столбца, ключа, группировки и соединения из 06.1. Все таблицы загрузим заново.
После пары вы сможете написать SELECT с условием и сортировкой, сгруппировать строки,
присоединить справочник и проверить результат запроса через Pandas. Никаких данных из чужого ядра.


### Подключим учебные файлы

        Следующая ячейка — готовая служебная настройка. Она ищет `course_support` в рабочей папке
        или выше. Её не надо переписывать по памяти. `DATA_DIR` указывает на учебные данные,
        `WORKSPACE_DIR` — на корень вашей копии, `check` запускает проверку упражнения.
        Если поддержка не найдена, перенесите весь комплект, а не только notebook.
        В этой тетради создаются все необходимые переменные: другую тетрадь запускать не нужно.


In [ ]:
import sys
from pathlib import Path

# Одинаковый запуск в репозитории курса и в личном проекте.
for WORKSPACE_DIR in (Path.cwd(), *Path.cwd().parents):
    if (WORKSPACE_DIR / 'course_support' / '__init__.py').is_file():
        break
else:
    raise FileNotFoundError('Откройте тетрадь внутри папки с course_support; нужен весь комплект проекта')

# При переключении курса в том же kernel не оставляем чужую поддержку в памяти.
loaded = sys.modules.get('course_support')
expected = WORKSPACE_DIR / 'course_support' / '__init__.py'
if loaded is not None and Path(loaded.__file__).resolve() != expected.resolve():
    for name in list(sys.modules):
        if name == 'course_support' or name.startswith('course_support.'):
            del sys.modules[name]
sys.path.insert(0, str(WORKSPACE_DIR))
from course_support import SUPPORT_DIR, DATA_DIR
COURSE_DIR = SUPPORT_DIR
# Только упражнения о личном README: в репозитории курса это ещё чистый шаблон.
PROJECT_DIR = WORKSPACE_DIR / 'project_template' if (WORKSPACE_DIR / 'project_template').is_dir() else WORKSPACE_DIR
from course_support.checks import check

PROJECT_DIR


### 1. Подготовим две таблицы

`sqlite3` — интерфейс Python к SQLite. `connect(':memory:')` открывает новую базу в памяти,
а переменная `connection` хранит подключение к ней. После закрытия это содержимое исчезнет;
исходные CSV останутся. `to_sql` копирует DataFrame в таблицу базы. `index=False` не записывает
технический индекс Pandas. `if_exists='replace'` заменяет одноимённую таблицу **этой временной базы**
при повторном запуске ячейки; так мы не добавим события дважды.


In [ ]:
import sqlite3
import pandas as pd
from IPython.display import display

listens = pd.read_csv(DATA_DIR / 'aggregation' / 'listens.csv', dtype={'event_id': 'string', 'track_id': 'string'})
tracks = pd.read_csv(DATA_DIR / 'aggregation' / 'tracks.csv', dtype={'track_id': 'string'})
connection = sqlite3.connect(':memory:')
listens.to_sql('listens', connection, index=False, if_exists='replace')
tracks.to_sql('tracks', connection, index=False, if_exists='replace')


`pd.read_sql_query(текст, подключение)` выполняет запрос и возвращает DataFrame.
Тройные кавычки Python позволяют записать многострочный текст. SQL внутри остаётся строкой,
пока мы не передадим его движку. `SELECT` выбирает столбцы, `FROM` — таблицу, из которой читаем.


In [ ]:
query = """
SELECT event_id, city, minutes
FROM listens
"""
result = pd.read_sql_query(query, connection)
display(result)


### 2. Добавляем одно условие

`WHERE` оставляет строки, для которых условие истинно. В SQL сравнение на равенство пишут `=`,
а не `==`; текстовое значение берут в одинарные кавычки. Названия столбцов здесь без кавычек.
Сначала только город — не добавляем сразу пять новых команд.


In [ ]:
query = """
SELECT event_id, city, minutes
FROM listens
WHERE city = 'Север'
"""
display(pd.read_sql_query(query, connection))


Теперь сузим вопрос: известная длительность хотя бы 3 минуты. `AND` соединяет два условия
словом «и»; `OR` — «или». Скобки помогают явно сгруппировать сложное условие.
Отсортируем через `ORDER BY minutes DESC`: DESC — по убыванию, ASC — по возрастанию.
`event_id ASC` разрешает равенства минут, чтобы первые строки были определены однозначно.


In [ ]:
query = """
SELECT event_id, city, minutes
FROM listens
WHERE city = 'Север' AND minutes >= 3
ORDER BY minutes DESC, event_id ASC
"""
display(pd.read_sql_query(query, connection))


Нужны только две верхние строки — в конце добавляем `LIMIT 2`. Без `ORDER BY` это
не «топ-2», а просто две строки без обещанного порядка. Не путайте ограничение показа и фильтр данных.


In [ ]:
query = """
SELECT event_id, minutes
FROM listens
WHERE city = 'Север' AND minutes >= 3
ORDER BY minutes DESC, event_id ASC
LIMIT 2
"""
display(pd.read_sql_query(query, connection))


### ✏️ Ваш ход — первый собственный запрос

Напишите `my_query`: выберите `event_id`, `minutes` для Юга,
где известно хотя бы 4 минуты. Отсортируйте минуты по убыванию, при равенстве — event_id по возрастанию.
Выполните через `pd.read_sql_query`, результат назовите `my_result`. До запуска выпишите ожидаемые ID.
Нужны все подходящие строки, LIMIT не требуется.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('06.2.1', my_result)


**Мой вывод:** …


### 3. Неизвестность в SQL называется NULL

Пустые длительности попали в базу как NULL. Проверка `minutes = NULL` не означает
«найти пустые»: сравнение с неизвестностью не даёт обычного истинного результата.
Для поиска есть `IS NULL`, для известных — `IS NOT NULL`. Ноль при этом остаётся числом.


In [ ]:
query = """
SELECT event_id, minutes
FROM listens
WHERE minutes IS NULL
ORDER BY event_id
"""
display(pd.read_sql_query(query, connection))


### 4. Одна строка на группу

`GROUP BY city` объединяет события города для расчёта показателя. `COUNT(*)` считает все строки,
`COUNT(minutes)` — известные длительности; `SUM` и `AVG` работают с известными числами.
`AS` задаёт имя столбца результата. Если известных значений нет, SUM/AVG в SQLite вернут NULL.
Пишем только ключи группировки и агрегаты: иначе смысл выбранной отдельной строки не определён нашей задачей.


In [ ]:
query = """
SELECT city,
       COUNT(*) AS events,
       COUNT(minutes) AS known_minutes,
       SUM(minutes) AS total_minutes,
       AVG(minutes) AS mean_minutes
FROM listens
GROUP BY city
ORDER BY city
"""
city_sql = pd.read_sql_query(query, connection)
display(city_sql)


Вернулись знакомые 19/21 минуты и разные знаменатели. `WHERE` отбирает **исходные строки**
до группировки. `HAVING` отбирает **готовые группы** по агрегату. Например, оставим города,
где известны хотя бы 5 длительностей. Это не отбор событий длиной хотя бы 5 минут.


In [ ]:
query = """
SELECT city, COUNT(minutes) AS known_minutes
FROM listens
GROUP BY city
HAVING COUNT(minutes) >= 5
ORDER BY city
"""
display(pd.read_sql_query(query, connection))


### ✏️ Ваш ход — не перепутать строки и группы

Получите `my_groups_sql` со столбцами `city`, `total_minutes`.
Нужны города, у которых сумма известных минут не меньше 20. Отсортируйте по city.
Напишите запрос с GROUP BY и HAVING. Объясните, почему WHERE minutes >=20 отвечает другому вопросу.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('06.2.2', my_groups_sql)


**Мой вывод:** …


### 5. Добавим справочник: LEFT JOIN

Слева события, справа жанры. `LEFT JOIN` сохраняет левую строку даже без пары.
Условие `ON listens.track_id = tracks.track_id` связывает ключи. При одинаковом имени столбца
пишем `таблица.столбец`, чтобы не было двусмысленности. Здесь намеренно без коротких псевдонимов:
сначала важно увидеть, откуда взялось каждое поле.


In [ ]:
query = """
SELECT listens.event_id, listens.track_id, listens.minutes, tracks.genre
FROM listens
LEFT JOIN tracks ON listens.track_id = tracks.track_id
ORDER BY listens.event_id
"""
joined_sql = pd.read_sql_query(query, connection)
display(joined_sql)


Никакого автоматического контроля «справа ключ уникален» SQL здесь не добавил:
`to_sql` создал простые учебные таблицы без ограничений ключа. Проверим справочник запросом.
Пустой результат значит «повторяющихся ключей не найдено», а не «запрос сломан».
В настоящей базе уникальность можно закрепить ограничением; проектирование схемы — отдельная тема.


In [ ]:
query = """
SELECT track_id, COUNT(*) AS rows_per_key
FROM tracks
GROUP BY track_id
HAVING COUNT(*) > 1
"""
display(pd.read_sql_query(query, connection))
print('События до/после:', len(listens), len(joined_sql))


### 6. Два способа — один проверяемый результат

Запишем одну задачу на Pandas. Методы уже разобраны: группа → сумма → обычный столбец города.
`reset_index(name='total_minutes')` задаёт имя столбца со значениями Series.
`.sort_values('city')` задаёт тот же порядок, `.reset_index(drop=True)` убирает старый технический индекс.
`pd.testing.assert_frame_equal` сравнивает две таблицы и сообщает различие; если всё совпало,
ничего не печатает. `check_dtype=False` разрешает различие текстовых типов Pandas/SQLite,
но значения, имена и порядок столбцов всё ещё проверяются.


In [ ]:
city_pd = listens.groupby('city')['minutes'].sum(min_count=1)
city_pd = city_pd.reset_index(name='total_minutes')
city_pd = city_pd.sort_values('city')
city_pd = city_pd.reset_index(drop=True)
query = """
SELECT city, SUM(minutes) AS total_minutes
FROM listens
GROUP BY city
ORDER BY city
"""
same_sql = pd.read_sql_query(query, connection)
pd.testing.assert_frame_equal(city_pd, same_sql, check_dtype=False)
print('Оба способа дали одинаковую таблицу')


### ✏️ Ваш ход — жанр через соединение

Напишите SQL-запрос с LEFT JOIN: `event_id`, `genre` для всех событий
площадки «Юг», отсортированных по event_id. Результат — `my_join_sql`.
Получите такую же таблицу через Pandas в `my_join_pd`: сначала merge с validate,
затем фильтр города, нужные столбцы, сортировка и reset_index(drop=True).
Сравните через `pd.testing.assert_frame_equal(..., check_dtype=False)`.
В результате должна остаться строка без жанра. Не заменяйте её вымышленной категорией.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('06.2.3', my_join_sql, my_join_pd)


**Мой вывод:** …


### Итог и следующий шаг проекта

Словарь получился небольшой: SELECT/FROM, WHERE, ORDER BY/LIMIT, GROUP BY/HAVING, LEFT JOIN/ON.
Главная работа всё та же: назвать смысл строки, выбрать показатель, сохранить неизвестность,
проверить ключи и результат. Совпадение двух способов помогает искать ошибки, но одинаково
неудачный вопрос они тоже могут посчитать одинаково.

Теперь можно переходить к [ЛР 3: очистка данных](../docs/ЛР 3.md), если вы открыли тетрадь в `workbooks`.
В корне репозитория документ находится в `docs/ЛР 3.md`. SQL-упражнения этой пары — тренировка;
они не добавляют новых критериев к ЛР 3. В проекте нужна воспроизводимая очистка и объяснение решений.
Срок объявляет преподаватель. GitHub необязателен: файлы и архив равноправны.

**За последние 5 минут:** выберите одну проверку для своего проекта. Напишите, какую ошибку она
ловит, и какой маленький испорченный пример должен её провалить. Это заготовка, не новая лабораторная.


### Дополнительно: значение из переменной

Если город пришёл из поля ввода, не собираем SQL склейкой строк. `?` — место параметра,
`params=(chosen_city,)` передаёт одно значение отдельно (запятая делает кортеж из одного элемента).
Движок трактует значение как данные. Параметры заменяют значения, а не имена таблиц.


In [ ]:
chosen_city = 'Юг'
query = "SELECT event_id, minutes FROM listens WHERE city = ? ORDER BY event_id"
display(pd.read_sql_query(query, connection, params=(chosen_city,)))


**Запас на 10–15 минут:** найдите события без справочника через `LEFT JOIN` и `WHERE tracks.track_id IS NULL`.
Затем отдельно найдите треки, которых никто не слушал: начните соединение с `tracks`.
Сначала предскажите разные ответы (t99 и t05), потом проверьте. Левую таблицу выбирают по вопросу.

Справочники: [SQLite в Python](https://docs.python.org/3/library/sqlite3.html),
[агрегаты SQLite](https://sqlite.org/lang_aggfunc.html). Обратите внимание на COUNT(*)/COUNT(X),
а не пытайтесь за вечер выучить все команды. Для закрепления достаточно одного собственного вопроса к таблицам.
